# Пакет `epi`: от папки с файлами `.e` до обучающей выборки

Демонстрация четырёх задач пайплайна:

1. **Каталог** — обойти папки с записями и собрать метаданные; краулер отслеживает изменения на диске.
2. **Данные пациента** — достать записи, метаданные, сигнал и маркеры конкретного человека.
3. **Обучающая выборка** — нарезать сигнал на окна и получить сбалансированный набор.
4. **Статистика** — выжимка по всей коллекции.

### Как запустить

Ноутбук должен работать на том же интерпретаторе, где стоит пакет. Проще всего запустить Jupyter прямо из виртуального окружения — тогда ядро `Python 3` и будет нужным интерпретатором:

```bash
cd /home/www1rt/Documents/EPI
./.venv/bin/jupyter lab demo.ipynb
```

Если вы предпочитаете запускать Jupyter из другого места, зарегистрируйте ядро один раз и выберите его в меню *Kernel → Change kernel*:

```bash
./.venv/bin/python -m ipykernel install --user --name epi --display-name "EPI (.venv)"
```

In [ ]:
import sys

import pandas as pd

import epi
from epi import Catalog, WindowSet, plot_marker, plot_window, render, summary

pd.set_option("display.width", 200, "display.max_colwidth", 60)

print("интерпретатор:", sys.executable)
print("версия epi:  ", epi.__version__)

## Задача 1. Каталог и краулер

`Catalog` — база SQLite с метаданными всей коллекции. Метод `sync()` обходит папку, разбирает новые файлы `.e` и приводит каталог в соответствие с диском.

Обход дешёвый: файл разбирается заново только если изменились его размер или время правки, **и** при этом разошёлся отпечаток содержимого. Поэтому повторный `sync()` на неизменных данных практически ничего не делает.

In [ ]:
catalog = Catalog("catalog.sqlite")

# первый вызов разбирает файлы, последующие только сверяют их с диском
counts = catalog.sync(".")
catalog

Все методы выборки возвращают списки словарей, поэтому их можно сразу передать в `pandas.DataFrame`.

Обратите внимание на `recordings` и `hours`: пациентов 16, а файлов 18 — у части людей несколько записей. Опознание идёт по GUID из заголовка файла, а не по имени папки, потому что имена папок содержат номера выгрузок. Одна и та же подпись `Patient1` на деле покрывала четырёх разных людей.

In [ ]:
patients = pd.DataFrame(catalog.patients())
print(catalog.totals())
patients[["patient_key", "alt_id", "dob", "recordings", "hours", "seizures"]].head(8)

## Задача 2. Данные и метаданные пациента

`catalog.patient()` собирает всё об одном человеке: поля из заголовка, список записей и все его приступы. Ключ можно указать началом GUID.

In [ ]:
key = patients.loc[0, "patient_key"]
info = catalog.patient(key)

print("пациент:       ", info["patient_key"])
print("дата рождения: ", info["dob"])
print("заметки врача: ", info["notes"])
print("приступов:     ", len(info["seizures"]))

pd.DataFrame(info["recordings"])[
    ["folder", "sampling_rate", "n_channels", "n_segments", "duration_sec"]
]

### От каталога к сигналу

`catalog.open()` возвращает `NicoletEReader` — это мост от строки каталога к сырым данным. Чтение оконное: с диска поднимается только запрошенный интервал, поэтому минута из многочасовой записи стоит несколько мегабайт.

Важная особенность формата: запись состоит из **сегментов**, разделённых настоящими разрывами во времени. Отсчёты читаются только внутри одного сегмента, поэтому `start_sec` отсчитывается от начала текущего сегмента.

In [ ]:
reader = catalog.open(info["recordings"][0]["id"])

print(reader)
print("каналы:  ", ", ".join(reader.channels))
print("сегменты:", reader.segments[:3])

# один канал за первую минуту, в микровольтах
ekg = reader.read_channel("EKG", start_sec=0, duration_sec=60)
print("\nEKG:", ekg.shape, ekg.dtype)

# несколько каналов сразу: массив (отсчёты, каналы)
block = reader.read(["Fp1", "Fp2", "C3", "C4"], start_sec=0, duration_sec=10)
print("блок из 4 каналов:", block.shape)

### Маркеры врача

Маркеры собираются из двух библиотек сразу, потому что каждая ошибается по-своему: `neo` верно раскладывает их по сегментам, но многие типы оставляет как `UNKNOWN` и портит кириллицу, а `pynicolet` называет больше типов и правильно декодирует текст, но в многосегментных файлах прижимает маркеры к концу записи. Ридер берёт от каждой то, что она делает верно.

Колонки `type_neo` и `type_pynicolet` показывают, что сказала каждая библиотека по отдельности — по ним видно, откуда взялось итоговое название.

In [ ]:
events = pd.DataFrame(reader.read_events())
print("маркеров в записи:", len(events))

events[["type", "text", "user", "segment", "segment_time_sec", "duration_sec",
        "type_neo", "type_pynicolet"]].head(10)

In [ ]:
# маркеры по всей коллекции, с фильтром по типу
seizures = catalog.events(type="Seizure")
print("приступов в коллекции:", len(seizures))

# красная линия — начало маркера, полоса — его длительность
plot_marker(catalog, seizures[0], pad_sec=5, window_sec=25);

## Задача 3. Обучающая выборка

`WindowSet` нарезает сигнал на окна одинаковой формы. Три решения продиктованы свойствами данных:

- окно целиком лежит **внутри одного сегмента** и не пересекает границу, потому что между сегментами настоящий разрыв во времени;
- разбиение делается **по пациентам**: окна одной записи перекрываются во времени и тривиально предсказываются друг из друга, поэтому случайное разбиение самих окон протащило бы тест в обучение;
- пациенты раздаются от самых богатых приступами к бедным, иначе фолд легко остаётся вовсе без положительных примеров.

Четвёртое решение продиктовано не данными, а тем, чему учится модель: насколько далеко от приступа должно стоять фоновое окно. Это настраивается отдельно и разбирается ниже.

In [ ]:
windows = WindowSet(catalog, length_sec=10.0, stride_sec=5.0, min_overlap_sec=5.0)

base_report = windows.build()   # разметить окна
print()
windows.split()   # раздать пациентов по фолдам
pd.DataFrame(windows.folds())

### Что окно знает о приступе

Разметка выше устроена просто: окно положительно, если в него попало не меньше `min_overlap_sec` секунд приступа, и отрицательно во всех остальных случаях. У такой простоты есть неприятное следствие — окно, снятое за секунду до приступа, и окно из середины спокойного часа получают одну и ту же метку «фон», хотя устроены совершенно по-разному.

Чтобы этим можно было управлять, при нарезке каждое окно считает расстояние до ближайшего приступа и хранит его в базе:

- `lead_sec` — сколько секунд от конца окна до начала следующего приступа;
- `lag_sec` — сколько секунд от конца предыдущего приступа до начала окна.

Пустое значение (`None`, в таблице pandas — `NaN`) означает, что с этой стороны приступов в сегменте нет вовсе: до первого приступа у окон нет `lag_sec`, после последнего — `lead_sec`. Окно, задевшее приступ, стоит от него на нуле с обеих сторон.

Ниже — окна вокруг первого размеченного приступа. Видно, как `lead_sec` убывает до нуля, затем идут окна с ненулевым `overlap_sec` (метку `1` из них получают те, где приступа набралось на `min_overlap_sec`), а после конца приступа начинает расти `lag_sec`.

In [ ]:
# отталкиваемся от окна с приступом и смотрим на его соседей по сегменту
hit = catalog.query("SELECT * FROM windows WHERE label = 1 LIMIT 1")[0]

near = catalog.query(
    """SELECT start_sec, label, overlap_sec, lead_sec, lag_sec FROM windows
       WHERE recording_id = ? AND segment = ? AND start_sec BETWEEN ? AND ?
       ORDER BY start_sec""",
    (hit["recording_id"], hit["segment"], hit["start_sec"] - 60, hit["start_sec"] + 60))

print("запись %d, сегмент %d, приступ около %.0f с от начала сегмента"
      % (hit["recording_id"], hit["segment"], hit["start_sec"]))
pd.DataFrame(near)

### Промежуток между окном и приступом

Ширина промежутка, отделяющего окно от приступа, задаётся при нарезке, раздельно с двух сторон:

- `gap_before_sec` — сколько секунд перед началом приступа считать загрязнёнными;
- `gap_after_sec` — то же после его конца.

Окно, попавшее в зазор, не годится ни в один класс: перед приступом сигнал уже меняется, после него держится постиктальное угнетение — назвать такое окно фоном значит научить модель на заведомо неверной метке. Оно получает метку `-1`, не попадает ни в фолды, ни в выборку, но остаётся в базе: `windows(label=-1)` покажет, что именно отброшено.

Стороны разведены намеренно. Постиктальный хвост тянется минутами, преиктальные изменения занимают гораздо меньший промежуток, поэтому одно число на обе стороны либо выбросит лишнего перед приступом, либо оставит грязь после него.

Положительных окон зазор не касается: приступ остаётся приступом, сколько бы вокруг него ни выбросили. Уменьшается только фон — поэтому доля положительных после такой нарезки растёт, хотя самих приступов не прибавилось.

По умолчанию оба зазора нулевые, и разметка получается ровно такой, какая была выше.

In [ ]:
guarded = WindowSet(catalog, length_sec=10.0, stride_sec=5.0, min_overlap_sec=5.0,
                    gap_before_sec=60.0, gap_after_sec=300.0)

guard_report = guarded.build()   # доля положительных считается от годных окон
print()
guarded.split()
pd.DataFrame(guarded.folds())

In [ ]:
# то, что ушло в зазор: приступа в окне нет, но и фоном его не назвать
grey = pd.DataFrame(guarded.windows(label=-1))

print("в зазоре %d окон из %d, в выборку пойдут %d"
      % (guard_report["excluded"], guard_report["total"], guard_report["usable"]))
grey[["start_sec", "label", "overlap_sec", "lead_sec", "lag_sec"]].head(8)

### Прогноз вместо обнаружения

Тот же промежуток меняет саму задачу. Если задать `preictal_sec`, положительными становятся не окна с приступом, а окна перед ним, и границы кладутся так:

- ближе `gap_before_sec` к началу приступа — исключено: за это время предупредить всё равно никто не успеет, а сигнал там уже почти приступ;
- от `gap_before_sec` до `gap_before_sec + preictal_sec` — положительный класс, окно ожидания;
- дальше — фон, если окно не задело постиктальный хвост предыдущего приступа.

Сам приступ уходит в исключённые: модель, которая учится по нему, учится обнаружению, а не прогнозу, и на отложенной выборке это выглядит как отличное качество при полной бесполезности.

Когда приступы идут кучно, окно может оказаться одновременно в постиктальном хвосте одного и в ожидании следующего. Исключение в таком споре сильнее: сомнительное окно лучше выбросить, чем отдать в положительный класс.

Числа ниже подобраны под эту коллекцию. В литературе окно ожидания обычно берут в полчаса, но здесь сегменты редко бывают настолько длинными — при получасовом горизонте размечать было бы почти нечего.

In [ ]:
predictive = WindowSet(catalog, length_sec=10.0, stride_sec=5.0,
                       gap_before_sec=60.0, gap_after_sec=300.0, preictal_sec=600.0)

pred_report = predictive.build()
print("\nположительный класс теперь --", predictive.positive_name)

pd.Series({"положительных": pred_report["positive"],
           "фоновых": pred_report["usable"] - pred_report["positive"],
           "исключено": pred_report["excluded"]}, name="окон").to_frame()

### Приступ и событие

До сих пор приступом считался маркер врача. Но маркеров на один приступ бывает несколько, а два разряда с короткой передышкой между ними — это скорее одно событие, чем два. Разметка «приступ — пауза — приступ» при этом учит модель искать границу там, где её нет: окна из паузы попадают в фон и оказываются почти неотличимы от положительных.

`merge_gap_sec` склеивает приступы, между которыми меньше указанного числа секунд, в одно событие — промежуток входит в него целиком. Даже при нулевом значении склеиваются пересекающиеся маркеры, иначе два маркера на один приступ посчитались бы дважды.

`positives_per_event` ограничивает вклад события в выборку. Соседние окна одного приступа перекрываются и отличаются сдвигом на шаг: десяток почти одинаковых примеров даёт модели ровно столько же, сколько один, но весит в десять раз больше — и это вес одного пациента в одну минуту его жизни. Представителем идёт окно, набравшее больше всего приступа (при разметке под прогноз — ближайшее к его началу), остальные уходят в исключённые вместе с теми, что событие лишь задели.

Номер события хранится у каждого окна в `event_idx`: у тех, что задели приступ, — его собственный, у остальных — ближайшего следующего, а если впереди приступов нет, то предыдущего.

In [ ]:
by_events = WindowSet(catalog, length_sec=10.0, stride_sec=5.0, min_overlap_sec=5.0,
                      merge_gap_sec=120.0, positives_per_event=1)

event_report = by_events.build()

columns = ("seizures", "events", "positive", "excluded")
pd.DataFrame([{"нарезка": "по маркерам", **{k: base_report[k] for k in columns}},
              {"нарезка": "по событиям", **{k: event_report[k] for k in columns}}]
             ).set_index("нарезка")

Разметка живёт в самой базе, и `build()` каждый раз переписывает таблицу окон целиком: в каталоге всегда лежит ровно один вариант — последний. Поэтому вернём детекторную разметку, чтобы выборка ниже собиралась из неё.

In [ ]:
windows.build(verbose=False)
windows.split(verbose=False)
windows

Приступы занимают около 3% сигнала, поэтому выборка набирается по классам отдельно, каждый до своей квоты. Заодно отбрасываются окна с плоским сигналом: усечённые выгрузки `Pruned` добиты сплошными нулями, и такие куски не должны попасть в обучение.

Сигнал приводится к 19 электродам схемы 10-20, которые есть в каждой записи, и к общей частоте 256 Гц — в коллекции есть и 500, и 512 Гц.

In [ ]:
x, y = windows.sample("train", n=64, positive_ratio=0.5, seed=0)

print("x:", x.shape, x.dtype, " (окна, каналы, отсчёты)")
print("y:", y.shape, " приступов:", int(y.sum()))
print("каналы:", ", ".join(windows.channels))

In [ ]:
# посмотреть глазами на то, что уехало в обучение
first_positive = int(y.argmax())
plot_window(x[first_positive], windows.channels, windows.rate, label=int(y[first_positive]));

### Из чего набирается фон

Фоновых окон в сотни раз больше, чем положительных, и случайная выборка почти вся приходит из спокойных часов: ровный сигнал, сон, ничего похожего на приступ. Модель на такой выборке учится лёгкой задаче — отличать приступ от сна, — а ошибается потом там, где сигнал на приступ похож.

`hard_ratio` задаёт долю фоновых окон, взятых вплотную к приступу: из полосы шириной `hard_span_sec`, начинающейся сразу за краем зазора, с обеих сторон. Здесь зазоры нулевые, поэтому трудными считаются все окна в пределах 600 секунд от приступа; при зазоре 60 с до и 300 с после это были бы окна за 60–660 секунд до приступа и через 300–900 секунд после него — остатки постиктального хвоста и преиктальный подъём.

Если трудных окон в фолде меньше запрошенного, недостача добирается обычным фоном: размер выборки от настроек не зависит, а печать показывает, сколько трудных нашлось на самом деле.

In [ ]:
x_hard, y_hard = windows.sample("train", n=64, positive_ratio=0.5,
                                hard_ratio=0.5, hard_span_sec=600.0, seed=0)

print("\nформа та же:", x_hard.shape, "| положительных:", int(y_hard.sum()))

In [ ]:
# то же самое, но на диск: внутри .npz лежат x, y, channels и rate
windows.export("val", out_dir="training", n=64, seed=0)

## Задача 4. Статистика

`summary()` собирает отчёт по каталогу вложенным словарём, `render()` печатает его по-человечески. Всё считается по базе, поэтому стоит миллисекунды и не трогает файлы `.e`.

In [ ]:
report = summary(catalog)
render(report)

In [ ]:
# отдельные разделы отчёта удобно смотреть таблицей
print("протоколы съёмки:")
display(pd.DataFrame(report["protocols"]))

print("\nтипы маркеров:")
display(pd.Series(report["markers"], name="штук").to_frame().head(10))

In [ ]:
import matplotlib.pyplot as plt

lengths = [e["duration_sec"] or 0 for e in catalog.events(type="Seizure")]

figure, axes = plt.subplots(figsize=(9, 3.5))
axes.hist([v for v in lengths if v <= 120], bins=40, color="steelblue")
axes.set_xlabel("длительность приступа, с")
axes.set_ylabel("маркеров")
axes.set_title("Приступы короче двух минут (%d из %d)"
               % (sum(v <= 120 for v in lengths), len(lengths)))
axes.spines[["top", "right"]].set_visible(False)
figure.tight_layout()

## Обновление датасета: добавление и удаление

Краулер сравнивает диск с каталогом. Повторный `sync()` на неизменных данных ничего не разбирает — все записи попадают в `unchanged`.

Записи, исчезнувшие с диска, помечаются удалёнными, но не стираются: так обучение можно проследить до тех данных, на которых оно шло. Из выборок они при этом сразу исчезают. Если файл вернётся на место, он опознается по отпечатку содержимого и снова станет живым (`revived`).

In [ ]:
catalog.sync(".")   # ничего не изменилось -> всё в unchanged

## То же самое из командной строки

```bash
python -m epi sync .                        # обновить каталог
python -m epi stats --json digest.json      # статистика, заодно в JSON
python -m epi patient <ключ>                # всё об одном пациенте
python -m epi index                         # разметить окна
python -m epi index --gap-before 60 --gap-after 300      # с зазором у приступа
python -m epi index --gap-before 60 --preictal 600       # разметка под прогноз
python -m epi index --merge-gap 120 --positives-per-event 1   # по окну на событие
python -m epi split                         # раздать пациентов по фолдам
python -m epi export --fold train --n 2000  # сохранить выборку в .npz
python -m epi export --fold train --hard-ratio 0.5      # фон вплотную к приступу
python -m epi plot --type Seizure --n 3     # нарисовать маркеры в .png
```

## Что стоит решить перед обучением

- **Длина окна.** При 10 секундах 38 сегментов короче одного окна выпадают целиком, а медиана длительности приступа — 5.5 секунды.
- **Ширина зазора у приступа.** По умолчанию она нулевая, и фоном считается всё, что не приступ, — включая окно, снятое за секунду до него. Сколько выбросить до и после, зависит от того, что должна выучить модель, и решать это стоит до обучения, а не после.
- **Что считать одним приступом.** Маркеров на один приступ бывает несколько, а два разряда с короткой передышкой — скорее одно событие. Порог склейки (`merge_gap_sec`) и вклад события в выборку (`positives_per_event`) задаются руками.
- **Состав фона.** Случайный фон почти весь приходит из спокойных часов. Долю окон, взятых вплотную к приступу, задаёт `hard_ratio` — от неё зависит, чему модель научится не путать с приступом.
- **Предобработка.** Сигнал отдаётся сырым, без фильтрации; фильтр и отбраковка артефактов в пакет не входят.
- **Размер валидации.** В `val` попали всего 2 пациента — для отбора модели это мало.
- **Аномалии разметки.** Самый долгий маркер приступа длится 1655 секунд (27 минут), что на приступ не похоже.